In [ ]:
import numpy as np
import pandas as pd
import os
import plotly.express as px
import scipy
import skimage

folder=r"J:\ctgroup\Edward\DATA\VMI\20260319\circular_3w_2p3e-7 PO\calibrated"

dfs={}

for file in os.listdir(folder):
    if file.endswith(".h5"):
        angle=int(file.split('_')[0])
        print(file)
        dfs[angle]=pd.read_hdf(os.path.join(folder,file))

In [ ]:
angle=-22
df=dfs[angle]
display(df.head())
df=df[df['m/q']>0]
df=df[df['m/q']<70]
fig=px.histogram(df, x='m/q',nbins=1000,title=f'Propylene Oxide Ion Spectrum',range_x=[0,70], height=800, width=1600
                 ).show()

In [ ]:
peaks=(14,27,42,55)
widths=2,5,5,5
full_dataframe=pd.DataFrame()
for i, (peak, width) in enumerate(zip(peaks,widths)):
    print(i,peak,width)
    for angle, df in dfs.items():
        df=df[df['m/q']>peak-width]
        df=df[df['m/q']<peak+width]
        df['angle']=angle
        df['ellipticity']=np.tan(np.radians(angle))
        df['fragment_id']=i
        full_dataframe=pd.concat([full_dataframe,df])

In [ ]:
full_dataframe=full_dataframe[full_dataframe['pz'].between(-1,1)]
full_dataframe.sort_values(by=['fragment_id','angle'],inplace=True)
px.density_heatmap(
        full_dataframe,
        x='py', y='pz',
        nbinsx=256, nbinsy=256,
        color_continuous_scale='Inferno',
        height=800, width=1600,
        facet_col='fragment_id', facet_row='angle',
        histnorm='probability density',
).show()

px.density_heatmap(
        full_dataframe,
        x='px', y='pz',
        nbinsx=256, nbinsy=256,
        color_continuous_scale='Inferno',
        height=800, width=1600,
        facet_col='fragment_id', facet_row='angle',
        histnorm='probability density',
)

In [ ]:
n_forward=full_dataframe[full_dataframe['py']>0].groupby(['fragment_id','angle']).size()
n_backward=full_dataframe[full_dataframe['py']<0].groupby(['fragment_id','angle']).size()
asymmetry=2*(n_forward-n_backward)/(n_forward+n_backward)
asymmetry=asymmetry.reset_index()
# asymmetry

full_dataframe["fba_contribution"]=full_dataframe.apply(lambda row: 1 if row['py']>0 else -1, axis=1)
full_dataframe['m/q bin']=pd.cut(full_dataframe['m/q'], bins=100)


left_df=full_dataframe[full_dataframe['angle']<0]
right_df=full_dataframe[full_dataframe['angle']>0]

left_df_fba=left_df.groupby('m/q bin')['fba_contribution'].mean().reset_index()
right_df_fba=right_df.groupby('m/q bin')['fba_contribution'].mean().reset_index()
left_df_fba['m/q']=left_df_fba['m/q bin'].apply(lambda x: x.left)
right_df_fba['m/q']=right_df_fba['m/q bin'].apply(lambda x: x.left)

px.line(
        left_df_fba,
        x='m/q', y='fba_contribution'
).add_scatter(
        x=right_df_fba['m/q'], y=right_df_fba['fba_contribution'], mode='lines'
)